# 🚀 AAMAS 2027: Session 1 — CIFAR-100 Personalization & Ablation Suite
### Kaggle Account 1 (27-Hour GPU Budget)

**Paper Target:** *Hierarchical Ensemble Personalization in Federated Learning (HEP-FL)* (AAMAS 2027)  
**Dataset:** **CIFAR-100** (Pre-attached in `/kaggle/input`, auto-linked in 0.0s)  
**Target Hardware:** Tesla T4 GPU (16 GB VRAM)  
**Assigned Jobs:**
1. **Job 1 (Table 1)**: Full Personalization Suite (7 Methods × 5 Dirichlet Regimes × 3 Seeds = 105 runs) at **100 rounds**.
2. **Job 5 (Table 5)**: Complete Ablation & Cluster Valuation Suite (5 Variants × 5 Regimes × 3 Seeds = 75 runs) at **100 rounds**.

> 💡 **Tip:** Set your Kaggle notebook accelerator to **GPU T4 x1**, then click **Save Version -> "Save & Run All (Commit)"** to run in the background without needing your browser open!


## 1. Environment Detection & Codebase Setup
Pulls the latest verified repository with fault-tolerant checkpointing and Kaggle dataset detection.


In [ ]:
import os, sys, shutil

ROOT = '/kaggle/working/Topology-aware-FDL'
OUT_DIR = '/kaggle/working/outputs'

if not os.path.exists(ROOT):
    print("🚀 Cloning repository from GitHub...")
    !git clone https://github.com/nam200718/Topology-aware-FDL.git {ROOT}
else:
    print("🔄 Repository exists, pulling latest updates from main...")
    !cd {ROOT} && git pull origin main

%cd {ROOT}
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print("✅ Current Working Directory:", os.getcwd())


## 2. Hardware Acceleration & Environment Verification
Verifies GPU acceleration and sets fast cuDNN primitives.


In [ ]:
!pip -q install pydantic pyyaml torchvision pandas matplotlib seaborn scipy
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM Capacity: {vram_gb:.2f} GB")
    torch.backends.cudnn.benchmark = True
else:
    print("⚠️ No GPU detected! Please navigate to Notebook Settings -> Accelerator -> GPU T4 x1.")


## 3. ⚡ Pre-loaded Kaggle Dataset Auto-Discovery (0.0s)
Directly verifies and symlinks `cifar-100-python` from `/kaggle/input` with zero network download.


In [ ]:
from src.data.dataset import _auto_link_dataset

data_dir = "./data"
found = _auto_link_dataset(data_dir, "cifar100")
if found:
    print("⚡ [Kaggle Input] CIFAR-100 dataset successfully verified and linked! (0.0s latency, 0 downloads)")
else:
    print("ℹ️ Note: If CIFAR-100 is attached under /kaggle/input, the experiment runner will automatically link it.")


## 4. 📊 Job 1: Main Heterogeneity & Personalization Benchmark (Table 1)
Evaluates all 7 methods across 5 Dirichlet non-IID regimes ($lpha \in [\infty, 1.0, 0.5, 0.1, 0.05]$) across 3 seeds (`42, 123, 7`) at **100 communication rounds**.
*Note: Uses atomic checkpointing — if the notebook stops or restarts, it automatically resumes without repeating finished runs.*


In [ ]:
# Run Job 1: Main Personalization Benchmark (Table 1)
!python -m src.baselines.run_all_baselines \
    --track personalization \
    --methods fedavg fedprox multikrum scaffold ditto fedrep hep_fl \
    --regimes iid mild moderate severe extreme \
    --rounds 100 \
    --seeds 42 123 7


### Inspect Job 1 Results Table


In [ ]:
import pandas as pd

p_csv = "outputs/baselines/personalization/results_personalization.csv"
if os.path.exists(p_csv):
    df_p = pd.read_csv(p_csv)
    print(f"✅ Job 1 completed successfully! Total evaluated cells: {len(df_p)}")
    display(df_p[["method", "regime", "mean_acc", "std_acc", "mean_b10", "std_b10", "elapsed_s"]])
else:
    print("⚠️ Checkpoint file not yet created.")


## 5. 🔬 Job 5: Ablation Study & Cluster Valuation Benchmark (Table 5)
Evaluates 5 architectural variants across 5 regimes across 3 seeds (`42, 123, 7`) at **100 communication rounds**:
1. `topo` (Proposed 3-tier HEP-FL)
2. `topo_no_aclm` (w/o Active-Class Logit Masking)
3. `topo_no_parent` (2-tier Root + Local, w/o Parent Head)
4. `topo_k1` (Grand Coalition, K=1)
5. `topo_oracle_k3` (Oracle label-distribution clustering Pareto upper bound)


In [ ]:
# Run Job 5: Ablation Study & Cluster Valuation (Table 5)
!python -m src.baselines.run_all_baselines \
    --track ablation \
    --methods topo topo_no_aclm topo_no_parent topo_k1 topo_oracle_k3 \
    --regimes iid mild moderate severe extreme \
    --rounds 100 \
    --seeds 42 123 7


### Inspect Job 5 Results Table


In [ ]:
a_csv = "outputs/baselines/ablation/results_ablation.csv"
if os.path.exists(a_csv):
    df_a = pd.read_csv(a_csv)
    print(f"✅ Job 5 completed successfully! Total evaluated cells: {len(df_a)}")
    display(df_a[["method", "regime", "mean_acc", "std_acc", "mean_b10", "std_b10", "elapsed_s"]])
else:
    print("⚠️ Checkpoint file not yet created.")


## 6. 📦 Packaging Account 1 Artifacts
Compresses all CSV and JSON results from Account 1 into a single zip file for 1-click download.


In [ ]:
import zipfile

zip_target = "/kaggle/working/account1_artifacts.zip"
with zipfile.ZipFile(zip_target, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk("outputs/baselines"):
        for file in files:
            full_p = os.path.join(root, file)
            rel_p = os.path.relpath(full_p, "outputs")
            zf.write(full_p, arcname=rel_p)

print(f"🎉 All Account 1 artifacts packaged into: {zip_target}")
print(f"File Size: {os.path.getsize(zip_target) / 1024:.1f} KB")
print("👉 You can now download 'account1_artifacts.zip' directly from the Kaggle Output file explorer on the right!")
